# 1 · Grandes volúmenes de datos con PySpark
**Torre del Caribe** · Universidad Nacional Rosario Castellanos · Licenciatura en Ciencia de Datos para Negocios, 5.º
semestre 2026-2 · Equipo 2

| | |
|---|---|
| **Materia (UCA)** | Herramientas para el Almacenamiento de Grandes Volúmenes de Datos |
| **Profesor** | Víctor Gabriel Cruz Arrona |
| **Criterio de la rúbrica** | 2. Gestión y almacenamiento de grandes volúmenes (12 %) |
| **Equipo** | Brandon Uriel García Sánchez · Maribel Mondragón Mercado · Jesús Ramírez Isidro · Enrique González Ortega |

El proyecto junta 18 fuentes oficiales distintas: archivos de texto con formato propio (huracanes),
JSON de API (clima, SITUR-Q), hojas de cálculo (DataTur), CSV enormes (6.1 millones de negocios del DENUE) y reseñas de
texto libre (Rest-Mex). Son datos **históricos y continuos, estructurados y semiestructurados**, que no coinciden entre
sí. En este notebook mostramos cómo los guardamos, limpiamos y consultamos con **PySpark**, **Parquet** y **DuckDB**, y
cómo la Torre en vivo los recibe semana a semana con **Spark Structured Streaming**.

| Parte | Qué respondemos |
|---|---|
| 1. La arquitectura medallón | ¿Dónde vive cada dato y cómo sabemos que no cambió? |
| 2. Spark | ¿Qué es y cómo se prepara en Windows? |
| 3. De Bronze a Silver con Spark | El archivo crudo de huracanes, convertido en tabla |
| 4. Consultas a escala | 6.1 millones de negocios y 542 mil horas de clima |
| 5. Reglas de calidad | Un cero no es un hueco; dos fuentes que no coinciden |
| 6. El almacén DuckDB | Modelo estrella y consultas en milisegundos |
| 7. Streaming | La Torre en vivo: una semana por lote |
| 8. Las columnas que inferimos | ¿Qué medimos y qué calculamos? De Bronze a las columnas _est |
| 9. Resumen de decisiones | Cada herramienta y parámetro, por qué y qué descartamos |

**Cómo leer este notebook.** Antes de cada celda de código explicamos qué hacemos y por qué; dentro del código, cada
renglón lleva su comentario. Al terminar cada paso comparamos nuestro resultado con el que guardó el proyecto (función
`coincide`): si una cifra no coincidiera, el notebook se detendría ahí. El documento técnico del equipo
(*Torre_del_Caribe.pdf*) tiene las fórmulas con su ejemplo resuelto a mano.

**Cómo correrlo.** Python 3.11 con `pip install -r requirements.txt` desde la carpeta de la entrega; abrir en Jupyter o
VS Code y elegir *Ejecutar todo*. Ya viene ejecutado, así que también se puede leer sin correrlo.

In [1]:
%matplotlib inline
# Preparación. Buscamos la carpeta de la entrega (la que tiene datos/silver), cargamos las librerías,
# definimos el estilo de las gráficas y la función que compara nuestras cifras con las del proyecto.
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib import font_manager

warnings.filterwarnings("ignore")                        # avisos de versiones de librerías que no cambian resultados
RAIZ = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "datos" / "silver").is_dir())
SILVER, GOLD = RAIZ / "datos" / "silver", RAIZ / "datos" / "gold"   # datos limpios y resultados del proyecto
FIGURAS = Path.cwd() / "figuras"                          # aquí guardamos cada gráfica como PNG
FIGURAS.mkdir(exist_ok=True)

# Colores institucionales de la UNRC (guinda y dorado) y una paleta para varias series.
GUINDA, DORADO, GRIS = "#9F2241", "#BC955C", "#3A3A3A"
PALETA = ["#9F2241", "#BC955C", "#565393", "#58A65D", "#8CAFDD", "#F26E50", "#465973"]
for f in (RAIZ / "herramientas" / "fuentes").glob("NotoSans-*.ttf"):
    font_manager.fontManager.addfont(str(f))          # la letra Noto Sans viene en la entrega
plt.rcParams.update({"font.family": "Noto Sans", "font.size": 10, "axes.spines.top": False, "axes.spines.right": False,
                     "axes.titleweight": "bold", "axes.titlecolor": GUINDA, "axes.titlesize": 12,
                     "text.color": GRIS, "axes.labelcolor": GRIS, "xtick.color": GRIS, "ytick.color": GRIS,
                     "figure.facecolor": "white", "savefig.dpi": 160, "savefig.bbox": "tight"})
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)


def coincide(nombre, nuestro, proyecto, tolerancia=1e-6):
    """Compara un resultado de este notebook con el que guardó el proyecto. Si no coincide, se detiene."""
    a = np.asarray(nuestro, dtype=float).ravel()
    b = np.asarray(proyecto, dtype=float).ravel()
    assert a.shape == b.shape, f"{nombre}: tamaños distintos {a.shape} y {b.shape}"
    dif = float(np.nanmax(np.abs(a - b))) if len(a) else 0.0
    print(f"{'✓' if dif <= tolerancia else '✗'} {nombre}: coincide con el proyecto (diferencia máxima {dif:.2g})")
    assert dif <= tolerancia, f"{nombre} no coincide con el proyecto"


def guardar(fig, nombre):
    """Guarda la gráfica en figuras/ con un nombre claro."""
    fig.savefig(FIGURAS / f"{nombre}.png")


print("Carpeta de la entrega:", RAIZ)

Carpeta de la entrega: C:\Users\Uriel\Desktop\PP 5to semestre CASCO SANTO TOMAS\Entregable


## 1. La arquitectura medallón: Bronze, Silver y Gold
| Capa | Qué guarda | Regla |
|---|---|---|
| **Bronze** | Los archivos tal como los publicó la fuente, en una carpeta por fecha de descarga | No se edita nunca. Cada archivo queda en un manifiesto con su huella **SHA-256** |
| **Silver** | Las mismas tablas limpias y tipadas, en **Parquet** particionado | Un hueco queda como nulo con su bandera; no se rellena |
| **Gold** | Los resultados de los modelos (índice, pronósticos, presupuesto...) | Cada tabla dice de qué Silver sale |

**Por qué así.** Si un número de la página sale raro, se puede seguir hacia atrás: Gold → Silver → el archivo exacto de
Bronze → la URL de donde se descargó. Y si una fuente cambia su archivo, la huella SHA-256 lo detecta.

**Por qué Parquet y no CSV.** Parquet guarda los tipos (una fecha es fecha, no texto), comprime por columna (el DENUE
pasa de 548 MB en CSV a 284 MB) y permite leer solo las columnas y las particiones que se necesitan.

La entrega trae completo Silver y Gold (360 MB) y, de Bronze, los archivos chicos que usan los notebooks; el Bronze
completo pesa 814 MB y se reconstruye con el código de descarga del proyecto.

In [2]:
def peso(carpeta):
    return sum(f.stat().st_size for f in carpeta.rglob("*") if f.is_file()) / 1e6

capas = []
for nombre, carpeta in [("silver", SILVER), ("gold", GOLD)]:
    for t in sorted(p for p in carpeta.iterdir() if p.is_dir() or p.suffix == ".parquet"):
        capas.append({"capa": nombre, "tabla": t.stem, "MB": round(peso(t) if t.is_dir() else t.stat().st_size / 1e6, 2),
                      "archivos": len(list(t.rglob("*.parquet"))) if t.is_dir() else 1})
capas = pd.DataFrame(capas)
print(capas.groupby("capa").agg(tablas=("tabla", "size"), MB=("MB", "sum")).round(1).to_string())
capas[capas.capa == "silver"].sort_values("MB", ascending=False)

        tablas     MB
capa                 
gold        52    3.3
silver      15  366.2


,capa,tabla,MB,archivos
6,silver,denue,293.88,576
13,silver,restmex,52.72,247
2,silver,clima_horario,8.15,83
1,silver,clima_diario,5.36,627
9,silver,huracanes,2.02,186
12,silver,nacionalidad,1.67,26
10,silver,inah,0.89,132
5,silver,datatur_ocupacion,0.67,8
0,silver,afac,0.20,22
7,silver,fred_diario,0.17,45


### El manifiesto: comprobar que el crudo no cambió
Cada descarga escribe un renglón en `MANIFIESTO.csv` con la fuente, el archivo, su **SHA-256**, su tamaño, la URL y la
hora de descarga. El SHA-256 es una huella de 64 caracteres: si cambia un solo byte del archivo, la huella cambia por
completo. Volvemos a calcularla para los archivos de Bronze que vienen en la entrega.

In [3]:
import hashlib

manifiesto = pd.read_csv(RAIZ / "datos" / "bronze" / "MANIFIESTO.csv")
print(f"{len(manifiesto)} archivos en el manifiesto, de {manifiesto.fuente.nunique()} fuentes:")
print(manifiesto.groupby("fuente").agg(archivos=("archivo", "size"), MB=("bytes", lambda b: round(b.sum() / 1e6, 1))).to_string())

def sha256(ruta):
    h = hashlib.sha256()
    with open(ruta, "rb") as f:
        for bloque in iter(lambda: f.read(1 << 20), b""):           # de 1 MB en 1 MB, sin cargar todo a memoria
            h.update(bloque)
    return h.hexdigest()

presentes = manifiesto[manifiesto.archivo.apply(lambda a: (RAIZ / a).exists())].copy()
presentes["sha256_hoy"] = presentes.archivo.apply(lambda a: sha256(RAIZ / a))
presentes["igual"] = presentes.sha256 == presentes.sha256_hoy
assert presentes.igual.all(), "un archivo crudo cambió desde su descarga"
print(f"\n✓ {len(presentes)} archivos de Bronze en la entrega: todos con la misma huella que el día de su descarga")
presentes[["fuente", "archivo", "bytes", "descargado_utc"]]

380 archivos en el manifiesto, de 18 fuentes:
                               archivos     MB
fuente                                        
D1 SITUR-Q                           14    3.2
D10 FRED                              2    0.2
D11 ENDUTIH 2025                      1    2.0
D12 GeoJSON Q. Roo                    1    1.5
D13 Benchmarks                        9   18.7
D18 SMN normales 1991–2020            3    0.0
D2 DataTur ocupación hotelera       166   67.8
D3 DataTur BD_Nacionalidad            1   19.0
D4 DataTur BdINAH                     1    3.1
D4 DataTur Compendio 2024             1   31.3
D4 DataTur DB_AFAC                    1    0.7
D4 DataTur cruceros                   1    0.2
D5 Rest-Mex 2025                      1   89.0
D6 DENUE INEGI                       33  574.2
D7 Censo 2020 ITER                    1    0.3
D8 Open-Meteo                       128   24.5
D9 HURDAT2 NOAA                       1    7.1
Evidencia sargazo                    15    8.8



✓ 14 archivos de Bronze en la entrega: todos con la misma huella que el día de su descarga


,fuente,archivo,bytes,descargado_utc
186,D9 HURDAT2 NOAA,datos/bronze/huracanes/2026-09-28/hurdat2-1851-2025-091226.txt,7071568,2026-09-28T18:44:32Z
190,D12 GeoJSON Q. Roo,datos/bronze/geo/2026-09-28/quintana_roo.json,1524156,2026-09-28T18:44:35Z
352,D13 Benchmarks,datos/bronze/benchmarks/2026-09-28/wordstream_google_2025.html,922412,2026-09-28T20:02:06Z
353,D13 Benchmarks,datos/bronze/benchmarks/2026-09-28/wordstream_google_2025.png,4656905,2026-09-28T20:02:06Z
354,D13 Benchmarks,datos/bronze/benchmarks/2026-09-28/wordstream_facebook_2025.html,913675,2026-09-28T20:02:14Z
355,D13 Benchmarks,datos/bronze/benchmarks/2026-09-28/wordstream_facebook_2025.png,3717931,2026-09-28T20:02:14Z
356,D13 Benchmarks,datos/bronze/benchmarks/2026-09-28/localiq_busqueda_2026.html,435635,2026-09-28T20:02:22Z
357,D13 Benchmarks,datos/bronze/benchmarks/2026-09-28/localiq_busqueda_2026.png,3324200,2026-09-28T20:02:22Z
358,D13 Benchmarks,datos/bronze/benchmarks/2026-09-28/localiq_facebook_2026.html,396015,2026-09-28T20:02:30Z
359,D13 Benchmarks,datos/bronze/benchmarks/2026-09-28/localiq_facebook_2026.png,4299683,2026-09-28T20:02:30Z


## 2. Spark: qué es y cómo se prepara
**Apache Spark** reparte el trabajo de una tabla grande entre varios procesos: parte los datos en **particiones**, cada
núcleo procesa las suyas y al final junta el resultado. Aquí corre en modo **local** con todos los núcleos de la
computadora; el mismo código correría sin cambios en un clúster de muchas máquinas, que es lo que pide el incidente
crítico de la materia ("cuando los datos no caben en una computadora").

Spark es **perezoso**: las transformaciones (`filter`, `withColumn`, `groupBy`) solo arman un plan; nada se calcula hasta
una acción (`count`, `show`, `write`). Eso le permite optimizar el plan completo, por ejemplo leer solo las columnas que
se usan.

**En Windows** necesita tres cosas: Java 17 (JAVA_HOME), `winutils.exe` y `hadoop.dll` para escribir Parquet
(HADOOP_HOME, en `herramientas/hadoop` de la entrega) y decirle qué Python usar. La carpeta tiene espacios en el nombre y
los scripts de Spark la cortan, así que usamos la "ruta corta" de Windows.

In [4]:
import glob, os, sys, time

def jdk17():
    candidatos = [c for p in (r"C:\Program Files\Microsoft\jdk-17*", r"C:\Program Files\Eclipse Adoptium\jdk-17*",
                              r"C:\Program Files\Java\jdk-17*") for c in glob.glob(p)]
    if not candidatos:
        raise RuntimeError("Falta Java 17: winget install Microsoft.OpenJDK.17")
    return max(candidatos, key=lambda r: tuple(int(n) for n in "".join(c if c.isdigit() else " " for c in Path(r).name).split()))

def ruta_corta(ruta):
    """La ruta sin espacios de Windows (PP 5to semestre... → PP5TOS~1); fuera de Windows no cambia nada."""
    if os.name != "nt":
        return ruta
    import ctypes
    b = ctypes.create_unicode_buffer(1024)
    n = ctypes.windll.kernel32.GetShortPathNameW(ruta, b, len(b))
    return b.value if 0 < n < len(b) else ruta

if os.name == "nt":
    os.environ["JAVA_HOME"] = jdk17()
    os.environ["HADOOP_HOME"] = str(RAIZ / "herramientas" / "hadoop")
    os.environ["PATH"] = os.pathsep.join([str(Path(os.environ["JAVA_HOME"]) / "bin"), str(RAIZ / "herramientas" / "hadoop" / "bin"), os.environ["PATH"]])
os.environ["PYSPARK_PYTHON"] = os.environ["PYSPARK_DRIVER_PYTHON"] = ruta_corta(sys.executable)

from pyspark.sql import SparkSession, Window, functions as F
spark = (SparkSession.builder.appName("notebook-grandes-volumenes")
         .master("local[*]")                                       # todos los núcleos de esta computadora
         .config("spark.driver.memory", "4g")                      # la máquina del equipo tiene 16 GB
         .config("spark.sql.session.timeZone", "UTC")              # las fechas no se mueven entre computadoras
         .config("spark.sql.shuffle.partitions", "8")              # pocas particiones al agrupar: los datos caben en una máquina
         .config("spark.ui.showConsoleProgress", "false")
         .getOrCreate())
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "· núcleos:", spark.sparkContext.defaultParallelism)

Spark 3.5.6 · núcleos: 12


## 3. De Bronze a Silver con Spark: el archivo de huracanes
HURDAT2 (NOAA) es un archivo de texto con un formato propio que **alterna** dos tipos de renglón: un **encabezado** por
tormenta (`AL092024, NADINE, 18,`: identificador, nombre y cuántos puntos siguen) y luego un renglón por cada punto de su
trayectoria, cada 6 horas. Un renglón de trayectoria no dice a qué tormenta pertenece: hay que "arrastrar" el encabezado
anterior.

In [5]:
archivo = sorted((RAIZ / "datos" / "bronze" / "huracanes").glob("*/hurdat2-1851-*.txt"))[-1]
with open(archivo, encoding="utf-8") as f:
    for _ in range(4):
        print(f.readline().rstrip())

AL011851,            UNNAMED,     14,
18510625, 0000,  , HU, 28.0N,  94.8W,  80, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999
18510625, 0600,  , HU, 28.0N,  95.4W,  80, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999
18510625, 1200,  , HU, 28.0N,  96.0W,  80, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999, -999


En Spark eso se resuelve con una **función de ventana**: numeramos los renglones en el orden del archivo y,
para cada renglón, tomamos el último encabezado visto hasta ahí (`last(..., ignorenulls=True)` sobre una ventana que va
del inicio al renglón actual). Luego sacamos posición, viento y presión con una **expresión regular**. Dos de los 55,524
renglones vienen mal formados en el archivo oficial (uno sin coma entre latitud y longitud, otro sin hemisferio): la
expresión tolera el primero, y en el segundo la latitud queda **nula**, no se adivina.

| Paso | Función de Spark | Qué hace |
|---|---|---|
| Leer | `spark.read.text` | Un renglón del archivo = una fila con la columna `value` |
| Ordenar | `monotonically_increasing_id` | Número creciente en el orden del archivo |
| ¿Encabezado? | `rlike("^[A-Z]{2}[0-9]{6},")` | Los encabezados empiezan con el identificador (AL092024) |
| Arrastrar | `last(ignorenulls=True).over(ventana)` | Cada punto hereda el identificador y el nombre de su tormenta |
| Extraer | `regexp_extract` | Latitud, longitud, viento y presión |
| Faltantes | `when(viento == -99, None)` | El −99 y el −999 oficiales son "sin dato", no números |

In [6]:
POS = r"(\d+\.\d)([NS])[\s,]*(\d+\.\d)([EW])\s*,\s*(-?\d+)\s*,\s*(-?\d+)"           # posición, viento, presión
SIN_HEMISFERIO = r"(\d+\.\d+)\s*,\s*(\d+\.\d)([EW])\s*,\s*(-?\d+)\s*,\s*(-?\d+)"    # la latitud de 1975 sin N/S
inicio = time.time()
crudo = spark.read.text(str(archivo)).withColumn("n", F.monotonically_increasing_id())
es_cab = F.col("value").rlike(r"^[A-Z]{2}[0-9]{6},")
partes = F.split("value", ",")
hasta_aqui = Window.orderBy("n").rowsBetween(Window.unboundedPreceding, 0)   # del primer renglón al actual
d = (crudo
     .withColumn("id_cab", F.when(es_cab, F.trim(partes[0])))
     .withColumn("nombre_cab", F.when(es_cab, F.trim(partes[1])))
     .withColumn("id_tormenta", F.last("id_cab", ignorenulls=True).over(hasta_aqui))      # arrastra el encabezado
     .withColumn("nombre", F.last("nombre_cab", ignorenulls=True).over(hasta_aqui))
     .filter(~es_cab)                                                                 # solo los puntos de trayectoria
     .withColumn("fecha_txt", F.trim(partes[0]))
     .withColumn("hora_utc", F.trim(partes[1]).cast("int"))
     .withColumn("bien", F.regexp_extract("value", POS, 1) != "")
     .withColumn("lat", F.when(F.col("bien"), F.regexp_extract("value", POS, 1).cast("double")
                               * F.when(F.regexp_extract("value", POS, 2) == "N", 1).otherwise(-1)))   # sin hemisferio: nula
     .withColumn("lon_txt", F.when(F.col("bien"), F.regexp_extract("value", POS, 3)).otherwise(F.regexp_extract("value", SIN_HEMISFERIO, 2)))
     .withColumn("hemi", F.when(F.col("bien"), F.regexp_extract("value", POS, 4)).otherwise(F.regexp_extract("value", SIN_HEMISFERIO, 3)))
     .withColumn("lon", F.col("lon_txt").cast("double") * F.when(F.col("hemi") == "W", -1).otherwise(1))
     .withColumn("viento_crudo", F.when(F.col("bien"), F.regexp_extract("value", POS, 5)).otherwise(F.regexp_extract("value", SIN_HEMISFERIO, 4)).cast("int"))
     .withColumn("viento_kt", F.when(F.col("viento_crudo") != -99, F.col("viento_crudo")))   # −99 = sin dato
     .withColumn("anio", F.substring("fecha_txt", 1, 4).cast("int"))
     .withColumn("mes", F.substring("fecha_txt", 5, 2).cast("int")))

# Distancia a Chetumal (haversine) con funciones de Spark, sin sacar los datos a Python
R, LAT0, LON0 = 6371.0, F.radians(F.lit(18.50)), F.radians(F.lit(-88.30))
a = (F.sin((F.radians("lat") - LAT0) / 2) ** 2 + F.cos(LAT0) * F.cos(F.radians("lat")) * F.sin((F.radians("lon") - LON0) / 2) ** 2)
d = (d.withColumn("km_chetumal", F.round(2 * R * F.asin(F.sqrt(a)), 1))
      .withColumn("afecta_sur_flag", (F.coalesce(F.col("km_chetumal"), F.lit(1e9)) <= 200)
                  & (F.coalesce(F.col("viento_kt"), F.lit(0)) >= 34) & (F.col("anio") >= 1966))
      .select("id_tormenta", "nombre", "anio", "mes", "fecha_txt", "hora_utc", "lat", "lon", "viento_kt", "km_chetumal", "afecta_sur_flag"))

SALIDA = Path.cwd() / "_spark" / "huracanes"
d.write.mode("overwrite").partitionBy("anio").parquet(str(SALIDA))                # Silver: Parquet partido por año
print(f"Leído, transformado y escrito en {time.time() - inicio:.1f} s")

Leído, transformado y escrito en 9.5 s


Comprobamos contra el Silver del proyecto (que se hizo con pandas): mismo número de puntos y tormentas,
mismas distancias y los mismos 31 eventos que afectan al sur.

In [7]:
nuestro = spark.read.parquet(str(SALIDA)).toPandas().sort_values(["id_tormenta", "fecha_txt", "hora_utc"]).reset_index(drop=True)
proyecto = (pd.read_parquet(SILVER / "huracanes", columns=["id_tormenta", "fecha", "hora_utc", "lat", "viento_kt", "km_chetumal", "afecta_sur_flag"])
            .assign(fecha_txt=lambda x: pd.to_datetime(x.fecha).dt.strftime("%Y%m%d"))
            .sort_values(["id_tormenta", "fecha_txt", "hora_utc"]).reset_index(drop=True))
print(f"{len(nuestro):,} puntos de {nuestro.id_tormenta.nunique():,} tormentas · latitudes nulas: {nuestro.lat.isna().sum()}")
coincide("número de puntos", len(nuestro), len(proyecto))
coincide("distancia a Chetumal", nuestro.km_chetumal.fillna(-1), proyecto.km_chetumal.astype(float).fillna(-1), tolerancia=0.051)
coincide("viento", nuestro.viento_kt.fillna(-1), proyecto.viento_kt.astype(float).fillna(-1))
coincide("tormentas que afectan al sur", nuestro[nuestro.afecta_sur_flag].id_tormenta.nunique(), 31)

55,524 puntos de 1,988 tormentas · latitudes nulas: 1
✓ número de puntos: coincide con el proyecto (diferencia máxima 0)
✓ distancia a Chetumal: coincide con el proyecto (diferencia máxima 0)
✓ viento: coincide con el proyecto (diferencia máxima 0)
✓ tormentas que afectan al sur: coincide con el proyecto (diferencia máxima 0)


La distancia puede diferir en 0.1 km en algún punto porque Spark redondea "hacia arriba" un 5 exacto y
Python redondea al número binario más cercano; por eso la tolerancia es de medio decímetro.

**Particiones.** Escribimos la tabla partida por año: una carpeta `anio=2024/` por año. Cuando una consulta filtra por
año, Spark **ni siquiera abre** las demás carpetas (*partition pruning*). Se ve en el plan de ejecución:

In [8]:
consulta = spark.read.parquet(str(SALIDA)).filter(F.col("anio") == 2024).groupBy("nombre").agg(F.max("viento_kt").alias("viento_max"))
import re
plan = consulta._jdf.queryExecution().executedPlan().toString()
print("Filtro de partición en el plan:", re.search(r"PartitionFilters: \[[^\]]*\]", plan).group(0))
print(f"Carpetas de año: {len(list(SALIDA.glob('anio=*')))} · la consulta de 2024 solo lee 1")
consulta.orderBy(F.desc("viento_max")).show(5)

Filtro de partición en el plan: PartitionFilters: [isnotnull(anio#271), (anio#271 = 2024)]
Carpetas de año: 175 · la consulta de 2024 solo lee 1


+------+----------+
|nombre|viento_max|
+------+----------+
|MILTON|       155|
| BERYL|       145|
|  KIRK|       130|
|HELENE|       120|
|RAFAEL|       105|
+------+----------+
only showing top 5 rows



## 4. Consultas a escala
### 6.1 millones de negocios (DENUE)
El Directorio Estadístico Nacional de Unidades Económicas del INEGI trae **todos los negocios del país**: 6,138,075
renglones. Es la tabla más grande del proyecto. La usamos para saber qué ofrece cada lugar (dónde comer, dónde dormir) y
para el planteamiento (cuánta oferta turística hay en cada municipio). Spark lee solo las columnas que pide la consulta.

In [9]:
inicio = time.time()
denue = spark.read.parquet(str(SILVER / "denue"))
total = denue.count()
por_estado = (denue.groupBy("entidad").agg(F.count("*").alias("negocios"), F.sum(F.col("es_turistico").cast("int")).alias("turisticos"))
              .withColumn("pct_turistico", F.round(100 * F.col("turisticos") / F.col("negocios"), 1))
              .orderBy(F.desc("pct_turistico")))
top = por_estado.limit(6).toPandas()
print(f"{total:,} negocios leídos y agrupados por estado en {time.time() - inicio:.1f} s")
coincide("renglones del DENUE", total, 6_138_075)
top

6,138,075 negocios leídos y agrupados por estado en 3.1 s
✓ renglones del DENUE: coincide con el proyecto (diferencia máxima 0)


,entidad,negocios,turisticos,pct_turistico
0,Nayarit,73357,15054,20.5
1,Quintana Roo,69260,13663,19.7
2,Colima,42073,8147,19.4
3,Baja California Sur,42577,7346,17.3
4,Aguascalientes,71871,12375,17.2
5,Durango,75111,12377,16.5


In [10]:
# Quintana Roo por municipio, con Spark SQL (la misma consulta escrita como SQL)
denue.createOrReplaceTempView("denue")
spark.sql("""
    SELECT municipio, COUNT(*) AS negocios, SUM(CAST(es_turistico AS INT)) AS turisticos,
           ROUND(100 * SUM(CAST(es_turistico AS INT)) / COUNT(*), 1) AS pct_turistico
    FROM denue WHERE es_qroo GROUP BY municipio ORDER BY turisticos DESC
""").toPandas()

,municipio,negocios,turisticos,pct_turistico
0,Benito Juárez,29633,5293,17.9
1,Solidaridad,10928,2204,20.2
2,Othón P. Blanco,10456,1836,17.6
3,Cozumel,5190,1148,22.1
4,Tulum,3619,1080,29.8
5,Bacalar,1406,422,30.0
6,Lázaro Cárdenas,1237,405,32.7
7,Felipe Carrillo Puerto,2660,402,15.1
8,Isla Mujeres,1518,385,25.4
9,Puerto Morelos,1344,319,23.7


Quintana Roo es de los estados con mayor proporción de negocios turísticos del país, y dentro del estado
la oferta se concentra en Benito Juárez (Cancún) y Solidaridad (Playa del Carmen): es la concentración que la campaña
quiere repartir.

### Ventanas de tiempo: 542 mil horas de clima
Del clima hora por hora (2019–2026, 8 puntos) sacamos la temperatura máxima de cada día y su **promedio móvil de 7 días**
con una ventana de Spark (`rowsBetween(-6, 0)`: el día y los 6 anteriores del mismo punto). Es la misma técnica que la
Torre en vivo usa para el promedio de 4 semanas de la ocupación del norte. Lo comprobamos contra pandas.

In [11]:
inicio = time.time()
hora = spark.read.parquet(str(SILVER / "clima_horario")).select("punto", "fecha_hora_local", "temp_c")
dia = hora.groupBy("punto", F.to_date("fecha_hora_local").alias("dia")).agg(F.max("temp_c").alias("temp_max"))
siete = Window.partitionBy("punto").orderBy("dia").rowsBetween(-6, 0)                  # el día y los 6 anteriores
dia = dia.withColumn("temp_max_7d", F.avg("temp_max").over(siete))
spark_7d = dia.toPandas().sort_values(["punto", "dia"]).reset_index(drop=True)
print(f"{hora.count():,} horas → {len(spark_7d):,} días con promedio móvil en {time.time() - inicio:.1f} s")

p = pd.read_parquet(SILVER / "clima_horario", columns=["punto", "fecha_hora_local", "temp_c"])  # lo mismo con pandas
p = p.groupby(["punto", p.fecha_hora_local.dt.date.rename("dia")]).temp_c.max().rename("temp_max").reset_index()
p["temp_max_7d"] = p.groupby("punto").temp_max.transform(lambda s: s.rolling(7, min_periods=1).mean())
coincide("promedio móvil de 7 días (Spark contra pandas)", spark_7d.temp_max_7d, p.temp_max_7d, tolerancia=1e-9)

542,784 horas → 22,624 días con promedio móvil en 1.5 s


✓ promedio móvil de 7 días (Spark contra pandas): coincide con el proyecto (diferencia máxima 1.4e-14)


## 5. Reglas de calidad
### Un cero no es un hueco
SITUR-Q publica sus series como una tabla por indicador. Cuando un mes no aparece, **no** quiere decir que llegaron cero
visitantes: quiere decir que no se publicó. En Silver ese mes queda con `hueco_flag = True` y valor nulo. Si lo
rellenáramos con cero, el pronóstico aprendería que en ese mes no llega nadie; si lo rellenáramos con el promedio,
estaríamos inventando un dato (regla de oro 1).

In [12]:
siturq = spark.read.parquet(str(SILVER / "siturq"))
(siturq.groupBy("indicador").agg(F.count("*").alias("meses"), F.sum(F.col("hueco_flag").cast("int")).alias("huecos"),
                                 F.sum(F.when(F.col("valor") == 0, 1).otherwise(0)).alias("ceros_reales"))
       .withColumn("pct_hueco", F.round(100 * F.col("huecos") / F.col("meses"), 1)).orderBy(F.desc("huecos")).toPandas())

,indicador,meses,huecos,ceros_reales,pct_hueco
0,ocupacion_hotelera,1953,693,0,35.5
1,aereos_llegadas,450,114,24,25.3
2,afluencia_turistas,600,15,0,2.5
3,derrama_visitantes,414,9,0,2.2
4,derrama_turistas,414,9,0,2.2
5,zonas_arqueologicas,910,0,166,0.0
6,habitaciones,825,0,0,0.0
7,centros_hospedaje,825,0,0,0.0
8,cruceristas,273,0,39,0.0
9,tren_maya_movimiento,248,0,2,0.0


### Cuando dos fuentes cuentan lo mismo
Los cruceristas aparecen en dos fuentes: DataTur (pasajeros por puerto) y SITUR-Q (cruceristas por destino). No
coinciden. No elegimos "la que nos conviene": las conciliamos mes a mes y año por año y medimos la diferencia. En
Cozumel, SITUR-Q cuenta entre −0.4 % y 15 % más en los años normales; en Mahahual, entre 11 % y 35 % más (los años de
pandemia se alejan más). Un mes suelto puede diferir mucho más (un mes de Cozumel difiere 2,725 %, por un cambio de
registro), por eso la comparación útil es anual. Cada uso del dato dice de qué fuente sale.

In [13]:
conc = pd.read_parquet(GOLD / "reconciliacion_cruceros.parquet").assign(anio=lambda x: pd.to_datetime(x.periodo).dt.year)
anual = conc.groupby(["puerto", "anio"])[["pasajeros_datatur", "cruceristas_siturq"]].sum()
anual["SITUR-Q contra DataTur (%)"] = (anual.cruceristas_siturq / anual.pasajeros_datatur - 1) * 100   # comparación anual
print("Diferencia mensual (%):", conc.groupby("puerto").diferencia_pct.agg(["min", "median", "max"]).round(1).to_dict("index"))
anual.round(1).unstack("puerto")["SITUR-Q contra DataTur (%)"]

Diferencia mensual (%): {'Cozumel': {'min': -12.4, 'median': 6.1, 'max': 2725.2}, 'Mahahual': {'min': -66.7, 'median': 19.1, 'max': 78.8}}


puerto,Cozumel,Mahahual
anio,,
2019,4.4,34.9
2020,13.3,31.5
2021,30.9,24.7
2022,15.0,24.0
2023,5.1,13.0
2024,-0.4,16.5
2025,4.0,11.0
2026,8.1,14.6


## 6. El almacén DuckDB y el modelo estrella
Spark limpia y escribe; para **consultar** en milisegundos no hace falta levantar Spark. **DuckDB** es una base de datos
analítica que vive en un archivo y lee los mismos Parquet **sin copiarlos**. Armamos el almacén aquí, con las rutas de
esta carpeta, para que funcione en cualquier computadora:

- una **vista** por cada tabla de Silver y Gold (si se reconstruye Silver, la vista ve el dato nuevo);
- el **modelo estrella**: `hechos_mes` (una medida por lugar y mes, en formato largo) con sus dimensiones `dim_lugar` y
  `dim_tiempo`. Formato largo porque cada fuente aporta las variables que tiene: un hueco simplemente no tiene renglón.

In [14]:
import duckdb

db = duckdb.connect()                                                # el almacén en memoria, con rutas de esta carpeta
for t in sorted(p.name for p in SILVER.iterdir() if p.is_dir()):
    db.execute(f"CREATE VIEW silver_{t} AS SELECT * FROM read_parquet('{(SILVER / t).as_posix()}/**/*.parquet', hive_partitioning = true, union_by_name = true)")
for t in sorted(GOLD.glob("*.parquet")):
    db.execute(f"CREATE VIEW gold_{t.stem} AS SELECT * FROM read_parquet('{t.as_posix()}')")

# Las tres tablas de la estrella son chicas: el proyecto las guarda materializadas en torre.duckdb y aquí las copiamos.
db.execute(f"ATTACH '{(GOLD / 'torre.duckdb').as_posix()}' AS proyecto (READ_ONLY)")
for nombre in ("dim_lugar", "dim_tiempo", "hechos_mes"):
    db.execute(f"CREATE TABLE {nombre} AS SELECT * FROM proyecto.{nombre}")
print(db.sql("SELECT variable, COUNT(*) AS renglones, MIN(periodo) AS desde, MAX(periodo) AS hasta FROM hechos_mes GROUP BY ALL ORDER BY 2 DESC").df().to_string(index=False))
print(f"Vistas: {len(db.sql('SELECT * FROM duckdb_views() WHERE NOT internal').fetchall())} · hechos_mes: {db.sql('SELECT COUNT(*) FROM hechos_mes').fetchone()[0]:,} renglones")

inicio = time.time()
r = db.sql("""
    SELECT CASE WHEN l.es_de_los_5 THEN 'los 5 del sur' ELSE 'resto del estado' END AS grupo,
           t.anio, ROUND(SUM(h.valor)) AS visitantes_inah
    FROM hechos_mes h JOIN dim_lugar l USING (lugar) JOIN dim_tiempo t USING (periodo)
    WHERE h.variable = 'visitantes_inah' AND t.anio BETWEEN 2022 AND 2025
    GROUP BY ALL ORDER BY grupo, t.anio
""").df()
print(f"Consulta de la estrella en {1000 * (time.time() - inicio):.0f} ms")
r.pivot(index="anio", columns="grupo", values="visitantes_inah")

                  variable  renglones      desde      hasta
                   hoteles        660 2022-01-01 2026-07-01
                   cuartos        660 2022-01-01 2026-07-01
llegadas_extranjeros_avion        498 2012-01-01 2026-07-01
           visitantes_inah        440 2022-01-01 2026-07-01
             ocupacion_pct        412 2022-01-01 2026-07-01
 pasajeros_crucero_datatur        254 2016-01-01 2026-07-01
             llegadas_tren        199 2024-01-01 2026-07-01
               cruceristas        110 2022-01-01 2026-07-01
             cruces_belice         54 2022-01-01 2026-06-01
Vistas: 122 · hechos_mes: 3,287 renglones
Consulta de la estrella en 8 ms


grupo,los 5 del sur,resto del estado
anio,,
2022,54910.0,1771684.0
2023,48218.0,1793004.0
2024,8770.0,1790839.0
2025,77645.0,1640778.0


La consulta cruza la tabla de hechos con sus dos dimensiones y responde en milisegundos: en 2025 las zonas
arqueológicas de los 5 lugares del sur recibieron 77,645 visitantes contra 1.64 millones del resto del estado (4.5 %);
en 2024 bajaron a 8,770 porque Oxtankah, Kohunlich y Dzibanché estuvieron cerradas por obras casi todo el año (el INAH reporta 0 visitantes). Es el desequilibrio que la campaña quiere
corregir.

**Por qué DuckDB y no PostgreSQL.** PostgreSQL exige instalar y levantar un servicio; el proyecto debe correr sin
internet en cualquier computadora (la del coloquio, la de un profesor). DuckDB es un archivo y una librería de Python.

## 7. Streaming: la Torre en vivo
El incidente crítico de la materia pide una arquitectura de **baja latencia** para ajustar la campaña **mientras corre**.
La Torre recibe una semana de señales a la vez (ocupación del norte, tormentas, clima, llegadas) y decide qué anuncio se
enciende. Para demostrarlo con datos reales, **reproducimos** las 239 semanas de 2022 a 2026 como si llegaran una por
una:

1. escribimos cada semana como un archivo JSON en una carpeta, en orden;
2. **Spark Structured Streaming** "escucha" esa carpeta y procesa **un archivo por lote** (`maxFilesPerTrigger = 1`);
3. en cada lote (`foreachBatch`) se toman las decisiones de esa semana.

Si mañana la carpeta la llenara una fuente real (o un tema de Kafka), el procesamiento no cambiaría. La celda tarda unos
6 minutos: Spark arranca un lote por semana, 239 veces.

In [15]:
import json, os, shutil

senales = pd.read_parquet(GOLD / "envivo_senales.parquet").sort_values("semana")
ENTRADA, PUNTO_CONTROL = Path.cwd() / "_spark" / "entrada", Path.cwd() / "_spark" / "punto_control"
shutil.rmtree(ENTRADA, ignore_errors=True); shutil.rmtree(PUNTO_CONTROL, ignore_errors=True); ENTRADA.mkdir(parents=True)
for i, fila in enumerate(senales.to_dict("records")):
    limpio = {k: (None if isinstance(v, float) and pd.isna(v) else v.isoformat() if hasattr(v, "isoformat") else v) for k, v in fila.items()}
    ruta = ENTRADA / f"semana_{fila['semana']:%Y-%m-%d}.json"
    ruta.write_text(json.dumps(limpio, ensure_ascii=False), encoding="utf-8")
    os.utime(ruta, (1_700_000_000 + 60 * i,) * 2)                   # hora creciente: Spark los toma del más viejo al más nuevo

recibido = []
def lote(df, id_lote):
    """Lo que hace la Torre con cada semana que llega (aquí: anotar la semana y si el norte está saturado)."""
    for fila in df.select("semana", "cancun_estado", "riviera_estado").collect():
        recibido.append({"lote": id_lote, "semana": fila.semana, "norte_saturado": "saturado" in (fila.cancun_estado, fila.riviera_estado)})

inicio = time.time()
esquema = spark.read.json(str(ENTRADA)).schema                       # el esquema se fija antes de escuchar
q = (spark.readStream.schema(esquema).option("maxFilesPerTrigger", 1).json(str(ENTRADA))
     .writeStream.foreachBatch(lote).option("checkpointLocation", str(PUNTO_CONTROL))
     .trigger(availableNow=True).start())                           # procesa lo que hay, lote por lote, y termina
q.awaitTermination()
rec = pd.DataFrame(recibido)
print(f"{len(rec)} semanas en {rec.lote.nunique()} lotes de Structured Streaming, en {time.time() - inicio:.0f} s")
assert rec.semana.is_monotonic_increasing, "las semanas no llegaron en orden"
coincide("semanas recibidas", len(rec), len(senales))
coincide("semanas con el norte saturado", rec.norte_saturado.sum(),
         ((senales.cancun_estado == "saturado") | (senales.riviera_estado == "saturado")).sum())
print("✓ las 239 semanas llegaron completas y en orden")

239 semanas en 239 lotes de Structured Streaming, en 233 s
✓ semanas recibidas: coincide con el proyecto (diferencia máxima 0)
✓ semanas con el norte saturado: coincide con el proyecto (diferencia máxima 0)
✓ las 239 semanas llegaron completas y en orden


El **punto de control** (*checkpoint*) guarda qué archivos ya se procesaron: si la Torre se cae a media
reproducción, al reiniciar sigue donde se quedó y no procesa dos veces la misma semana. Las decisiones completas de cada
semana (encender, pausar, "¿Ibas al norte?") se reconstruyen en el notebook 6.

**Privacidad.** Ninguna fuente del proyecto trae datos personales: todo son agregados oficiales (por mes, por lugar, por
negocio) o reseñas anónimas. La página guarda en Firebase solo un contador de corazones por foto y, en "Cotiza tu viaje",
los datos de contacto que la persona decide enviar, con su consentimiento y su aviso de privacidad.

## 8. Las columnas que inferimos: de dónde sale cada una
Una pregunta que nos van a hacer: **¿qué columnas medimos y cuáles calculamos nosotros?** La regla del proyecto es que
todo lo estimado lleva el sufijo `_est` (regla de oro 3). Con DuckDB podemos leer el **esquema** de todos los Parquet de
Gold de un golpe, sin cargar los datos, y listar cada columna inferida con la tabla donde vive.

In [16]:
import duckdb
inferidas = duckdb.sql(f"""
    SELECT regexp_extract(file_name, '([a-z_]+)\.parquet', 1) AS tabla, name AS columna
    FROM parquet_schema('{(GOLD / "*.parquet").as_posix()}')
    WHERE name LIKE '%_est'
    ORDER BY tabla, columna""").df()
print(len(inferidas), "columnas inferidas en", inferidas.tabla.nunique(), "tablas de Gold")
inferidas

24 columnas inferidas en 9 tablas de Gold


,tabla,columna
0,presupuesto_montecarlo,campana_p10_est
1,presupuesto_montecarlo,campana_p50_est
2,presupuesto_montecarlo,campana_p90_est
3,presupuesto_montecarlo_resumen,campana_p10_est
4,presupuesto_montecarlo_resumen,campana_p50_est
5,presupuesto_montecarlo_resumen,campana_p90_est
6,presupuesto_plan,conversiones_est
7,pronostico_calendario,esperado_est
8,pronostico_calendario,maximo_90_est
9,pronostico_calendario,minimo_90_est


Cada una tiene una cadena que se puede seguir hacia atrás hasta un archivo de Bronze:

| Columna inferida | Tabla (Gold) | Qué es | De qué sale | Cómo se validó |
|---|---|---|---|---|
| `estado_mes_siguiente_est` y `prob_*` | `radar_prediccion` | Estado del mes siguiente y su probabilidad | Rasgos del índice de presión (Tren Maya, SITUR-Q, INAH, DataTur, Censo) → Random Forest + año anterior | 130 de 156 en la prueba; Brier 0.214 |
| `esperado_est`, `minimo_90_est`, `maximo_90_est` | `pronostico_mes` | Visitantes esperados y su rango del 90 % | Serie mensual (INAH, SITUR-Q, DataTur) → modelo elegido + cuantil conformal | MAPE 11–22 %; cobertura medida 80–94 % |
| `ocupacion_est` | `pronostico_calendario` | Ocupación hotelera esperada del norte | DataTur → modelo elegido | MAPE 3.8 % (Cancún) |
| `malo_p10_est`, `probable_p50_est`, `bueno_p90_est` | `pronostico_escenarios` | Escenarios malo, probable y bueno | Pronóstico × errores reales de un año (Monte Carlo, 10,000) | Desde 5,000 futuros ya no cambia |
| `conversiones_est` | `presupuesto_plan` | Visitantes que trae cada peso | Pesos (PuLP) × conversión ÷ (costo por clic × tipo de cambio) | Benchmarks WordStream y LocaliQ; FRED |
| `campana_p10/p50/p90_est` | `presupuesto_montecarlo` | Rango de lo que trae la campaña | 10,000 sorteos de los supuestos dentro del rango de las fuentes | 82 % de probabilidad de traer menos que el plan |
| `semanas_al_anio_est` | `radar_markov_largo_plazo` | Semanas saturadas al año (52 · π) | Distribución estacionaria de la cadena de Markov (DataTur semanal) | π = 9.8 % contra 10 % observado |

Ahora comprobamos **con código** tres de esas cadenas, de punta a punta. Primero, con Spark: la serie de la Ruta en Gold
debe ser exactamente la suma de las tres zonas (Kohunlich, Dzibanché e Ichkabal) en Silver, mes por mes.

In [17]:
inah = spark.read.parquet(str(SILVER / "inah"))
ruta_spark = (inah.filter((F.col("estado") == "Quintana Roo") & F.col("nombre").rlike("Kohunlich|Dzibanch|Ichkabal"))
                  .groupBy("periodo").agg(F.sum("visitantes").alias("suma_silver"))     # nacionales + extranjeros, 3 zonas
                  .toPandas())
ruta_spark["periodo"] = pd.to_datetime(ruta_spark.periodo)
serie = pd.read_parquet(GOLD / "pronostico_series.parquet")
serie = serie[serie.lugar == "Ruta arqueológica del sur"][["periodo", "valor"]]
cadena = serie.merge(ruta_spark, on="periodo")
print(f"{len(cadena)} meses comparados · diferencia máxima Gold − Silver: {(cadena.valor - cadena.suma_silver).abs().max():.0f}")
assert (cadena.valor == cadena.suma_silver).all(), "la serie de Gold no es la suma de Silver"
cadena.tail(4)

127 meses comparados · diferencia máxima Gold − Silver: 0


,periodo,valor,suma_silver
123,2026-04-01,5961.0,5961
124,2026-05-01,3842.0,3842
125,2026-06-01,3126.0,3126
126,2026-07-01,5057.0,5057


Segundo, el estado del Radar: debe salir del índice con los dos cortes, sin excepción. Y tercero, las
conversiones del plan: el cociente `conversiones_est ÷ pesos` debe ser **una sola constante por canal** (el rendimiento
de un peso), porque la conversión es lineal en el gasto.

In [18]:
ic = pd.read_parquet(GOLD / "radar_indice_comparable.parquet").dropna(subset=["ipt_comparable"])
recalculado = np.select([ic.ipt_comparable < ic.corte_p50, ic.ipt_comparable <= ic.corte_p90],
                        ["tranquilo", "concurrido"], default="saturado")
print("estado del Radar recalculado igual en", int((recalculado == ic.estado).sum()), "de", len(ic), "lugar-meses")
assert (recalculado == ic.estado).all()
plan = pd.read_parquet(GOLD / "presupuesto_plan.parquet")
plan = plan[plan.pesos > 0.01]
rendimiento = (plan.conversiones_est / plan.pesos).groupby(plan.canal).agg(["min", "max"])
print("visitantes por cada $1,000, por canal:")
print((rendimiento * 1000).round(4).to_string())
assert (rendimiento["max"] - rendimiento["min"]).max() < 1e-12      # una sola constante por canal

estado del Radar recalculado igual en 510 de 510 lugar-meses
visitantes por cada $1,000, por canal:
             min     max
canal                   
Facebook  6.6084  6.6084
Google    1.5898  1.5898


Las tres cadenas cierran: la serie de Gold es la suma exacta de Silver, cada estado sale de su índice y
cada conversión es pesos × rendimiento del canal. Las capturas de estos DataFrames, de Bronze a las columnas `_est`,
están en el capítulo 3 del documento técnico.

In [19]:
spark.stop()
shutil.rmtree(Path.cwd() / "_spark", ignore_errors=True)            # limpiamos lo que escribimos en esta carpeta
print("Spark detenido")

Spark detenido


## 9. Resumen de decisiones

| Decisión | Elegimos | Por qué | Qué descartamos |
|---|---|---|---|
| Arquitectura | Medallón Bronze / Silver / Gold | Cada cifra se rastrea hasta su archivo y su URL | Una sola carpeta de datos "limpios" |
| Integridad | Manifiesto con SHA-256 | Detecta cualquier cambio de un byte | Confiar en la fecha del archivo |
| Formato | Parquet particionado por año | Tipos, compresión por columna, lectura parcial | CSV (sin tipos, 2× más pesado) |
| Motor | PySpark 3.5 en modo `local[*]` | El mismo código escala a un clúster | pandas solo (no escala); Dask (menos usado en la industria) |
| Memoria | `spark.driver.memory = 4g` | La máquina del equipo tiene 16 GB | El valor por defecto (1 GB: el DENUE no cabe al agrupar) |
| Zona horaria | UTC | Las fechas no se mueven entre computadoras | La hora local de cada máquina |
| Particiones al agrupar | 8 | Los datos caben en una máquina; 200 (por defecto) crea muchas tareas vacías | 200 |
| Huecos | Nulo + `hueco_flag` | No se inventan datos | Rellenar con 0 o con el promedio |
| Almacén | DuckDB con vistas y modelo estrella | Archivo local, consultas en milisegundos, sin servidor | PostgreSQL (servicio que instalar) |
| Columnas inferidas | Sufijo `_est` y cadena comprobada con código | Se distingue lo medido de lo estimado | Mezclar estimados con datos sin marcarlos |
| Tiempo casi real | Structured Streaming, 1 archivo por lote, `availableNow` | Misma arquitectura que con datos en vivo; se puede reanudar | Un ciclo `for` (no es streaming); Kafka (no hay fuente en vivo que conectar) |
| Java en Windows | JDK 17 solo para este proceso | No cambia el Java del sistema | Docker (más pesado de arrancar el día del coloquio) |

**Qué decisión alimenta.** Todo lo demás: sin Bronze trazable, Silver limpio y Gold documentado, ninguna cifra de la
campaña se podría defender. Y la arquitectura de streaming es la que permite pausar un anuncio la misma semana en que
llega una tormenta.